### 新客、老客客诉宽表


In [ ]:

DROP TABLE IF EXISTS lss_order_loan_tousu;
CREATE TABLE lss_order_loan_tousu AS

-- 首贷
WITH
-- 日期周期参考表 
 cycle_ref AS
(
	SELECT  a.day_id_iso                                                  AS single_day
	       ,b.day_id_iso                                                  AS cycle_end_date
	       ,DATEDIFF(to_date(b.day_id_iso),to_date(a.day_id_iso),'dd')/30 AS mob
	FROM
	(
		SELECT  day_id_iso
		       ,1 AS tmp
		FROM xyf_dim.dim_pub_date
	) a
	INNER JOIN
	(
		SELECT  day_id_iso
		       ,1 AS tmp
		FROM xyf_dim.dim_pub_date
	) b
	ON a.day_id_iso >= '2025-01-01' AND a.day_id_iso <= DATE(getdate()) AND b.day_id_iso <= DATE(getdate())
	AND a.tmp = b.tmp AND DATEDIFF(to_date(b.day_id_iso), to_date(a.day_id_iso), 'day') IN (30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 360, 390, 420, 450, 480, 510, 540)
	ORDER BY a.day_id_iso, b.day_id_iso
)

-- 授信数据 
 , shouxin_xujia_info AS (
SELECT  shouxin.*
       ,CASE WHEN shouxin.授信渠道 IN ('APP','微信小程序授信','抖音小程序授信') AND xujia.biz_flow_number IS NOT NULL THEN 'APP虚假给额'
             WHEN shouxin.授信渠道 = 'API半流程' AND shouxin.授信金额 < 1000 THEN '半流程虚假给额' --半流程的虚假给额规则 
             WHEN shouxin.授信渠道 = 'API' AND DATE(shouxin.授信时间) >= '2025-12-01' AND shouxin.授信金额 < 1000 THEN 'API虚假给额'  
	    ELSE '非虚假给额' END AS is_虚假给额 
			 -- API、API半流程都是12月之后左右开始有虚假给额的情况出现，API的虚假给额在渠道端显示为失败，但是短信会引导用户回APP完成办卡借款 
FROM
(
	SELECT  user_no
	       ,cust_no
	       ,init_credit_line / 100 AS 授信金额 --单位：分 
	       ,created_time           AS 授信时间
	       ,credit_success_time
	       ,credit_expire_date
	       ,inner_app
	       ,biz_flow_number
	       ,CASE WHEN lower(inner_app) IN ('xyf01','fxk','cxh') AND client_code IN ('MPP001000068') THEN '微信小程序授信'
	             WHEN lower(inner_app) IN ('xyf01','fxk','cxh') AND client_code IN ('MPP002000069') THEN '抖音小程序授信'
	             WHEN lower(inner_app) IN ('xyf01','fxk','cxh') THEN "APP"
	             WHEN lower(inner_app) IN ('xyf01_hrui02','xyf01_xcjr','xyf01_hrui01','xyf01_alyxy','xyf01_alygd','xyf01_alyfz','xyf01_zyxj01','xyf01_zyxjwld01','xyf01_zyxjzl01','xyf01_elm') THEN 'API半流程'  
				 ELSE 'API' END AS 授信渠道
	FROM xyf_dwd.dwd_preloan_credit_apply_df
	WHERE pt = '${bizdate}'
	AND app IN ('xyf01', 'fxk')
	AND status = 2 --授信成功 
) shouxin
LEFT JOIN
( -- 虚假给额的授信成功用户口径，biz_flow_number关联授信表 
	SELECT  biz_flow_number -- 授信biz_flow_number 
	FROM xyf_dwd.dwd_inloan_t_decision_result_detail_df
	WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_t_decision_result_detail_df')
	AND enginecode = 'jcl_20240923000003'
	AND GET_JSON_OBJECT(context, "$.app_new_risk_mark_output") RLIKE 'fake_activation'
	AND decision_time >= '2024-10-15 00:00:00' --
	-- 存在少量异常数据，同一个biz_flow_number+enginename 存在多条记录；下面排序做个兜底的清洗 
 QUALIFY ROW_NUMBER() OVER (PARTITION BY biz_flow_number, date(decision_time) ORDER BY decision_time DESC ) = 1 
) xujia
ON shouxin.biz_flow_number = xujia.biz_flow_number 
) , 

-- 首贷基础放款数据 
first_loan_info AS 
 (
SELECT  shoudai_order.*
       ,shouxin_xujia_info.授信金额  
       ,CASE WHEN shouxin_xujia_info.授信金额 <= 1000 THEN '[0-1000]'
             WHEN shouxin_xujia_info.授信金额 <= 2000 THEN '(1000-2000]'
             WHEN shouxin_xujia_info.授信金额 <= 3000 THEN '(2000-3000]'
             WHEN shouxin_xujia_info.授信金额 <= 5000 THEN '(3000-5000]'
             WHEN shouxin_xujia_info.授信金额 <= 10000 THEN '(5000-10000]'
             WHEN shouxin_xujia_info.授信金额 <= 20000 THEN '(10000-20000]'
             WHEN shouxin_xujia_info.授信金额 <= 50000 THEN '(20000-50000]'
             WHEN shouxin_xujia_info.授信金额 <= 100000 THEN '(50000-100000]'
             WHEN shouxin_xujia_info.授信金额 > 100000 THEN '(100000+]'  ELSE '' END AS 授信额度区间
       ,shouxin_xujia_info.授信时间
       ,shouxin_xujia_info.inner_app                                             AS 授信inner_app
       ,shouxin_xujia_info.user_no                                               AS 授信user_no
       ,shouxin_xujia_info.授信渠道
       ,shouxin_xujia_info.is_虚假给额
FROM
(
	SELECT  DATE(first_order_time)       AS 订单发起日期
	       ,SUBSTR(first_order_time,1,7) AS 订单发起月
	       ,DATE(loan_time)              AS 放款日期
	       ,SUBSTR(loan_time,1,7)        AS 放款月
	       ,order_number
	       ,user_no
	       ,cust_no
	       ,first_order_number
	       ,app
	       ,inner_app
	       ,business_line
	       ,loan_flag
	       ,first_order_time
	       ,loan_time
	       ,loan_amt
	       ,period
	       ,asset_type_flag
	       ,fee_rate
	       ,biz_flow_number
	FROM xyf_dws.dws_inloan_user_order_df
	WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
	AND DATE(first_order_time) >= '2025-01-01'
	AND app IN ('xyf01', 'fxk')
	AND business_line IN ('APP', '小程序端') -- APP首贷订单 
	AND loan_status = 'success'
	AND loan_flag = '首贷' 
) shoudai_order
LEFT JOIN shouxin_xujia_info
ON shoudai_order.cust_no = shouxin_xujia_info.cust_no AND shoudai_order.first_order_time >= shouxin_xujia_info.授信时间 --偶见一些用户user_no1授信，然后用user_no2借款的情况，因此用cust_no关联 
 AND shouxin_xujia_info.credit_expire_date > shoudai_order.first_order_time AND shouxin_xujia_info.credit_success_time <= shoudai_order.first_order_time 
 QUALIFY ROW_NUMBER() OVER (PARTITION BY shoudai_order.order_number ORDER BY shouxin_xujia_info.授信时间 DESC ) = 1 --取首贷订单发生前最近一次授信记录 
),

-- 首贷,复贷基础放款数据合并
loan_orders_info AS
(

SELECT  *
       ,CAST(NULL AS DECIMAL(38,18)) AS 已用额度
       ,CAST(NULL AS STRING)         AS 已用额度区间

FROM first_loan_info

UNION ALL

SELECT  fudai_order.*
       ,first_loan_info.授信金额
	   ,first_loan_info.授信额度区间
	   ,first_loan_info.授信时间
	   ,first_loan_info.授信inner_app
	   ,first_loan_info.授信user_no
	   ,first_loan_info.授信渠道
	   ,first_loan_info.is_虚假给额
       ,base.已用额度
       ,base.已用额度区间 
FROM
(
	SELECT  DATE(first_order_time)       AS 订单发起日期
	       ,SUBSTR(first_order_time,1,7) AS 订单发起月
	       ,DATE(loan_time)              AS 放款日期
	       ,SUBSTR(loan_time,1,7)        AS 放款月
	       ,order_number
	       ,user_no
	       ,cust_no
	       ,first_order_number
	       ,app
	       ,inner_app
	       ,business_line
	       ,'复贷' AS loan_flag
	       ,first_order_time
	       ,loan_time
	       ,loan_amt
	       ,period
	       ,asset_type_flag
	       ,fee_rate
	       ,biz_flow_number
	FROM xyf_dws.dws_inloan_user_order_df
	WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
	AND DATE(first_order_time) >= '2025-01-01'
	AND app IN ('xyf01', 'fxk')
	AND business_line IN ('APP', '小程序端') -- APP复贷订单 
	AND loan_status = 'success'
	AND loan_flag <> '首贷' 
) fudai_order
LEFT JOIN
(
	SELECT  user_no
	       ,cust_no
		   ,line_used_amt                                                                AS 已用额度  --用户借款已累计使用额度情况
	       ,CASE WHEN line_used_amt  < 500 THEN '1.[0,500)'
	             WHEN line_used_amt  < 1000 THEN '2.[500,1000)'
	             WHEN line_used_amt < 3000 THEN '3.[1000,3000)'
	             WHEN line_used_amt < 6000 THEN '4.[3000,6000)'
	             WHEN line_used_amt < 9000 THEN '5.[6000,9000)'
	             WHEN line_used_amt < 12000 THEN '6.[9000,12000)'
	             WHEN line_used_amt < 15000 THEN '7.[12000,15000)'
	             WHEN line_used_amt >= 15000 THEN '8.[15000,15000+)'  ELSE 'unknown' END AS 已用额度区间  
	       ,DATE(TO_DATE(pt,'yyyymmdd')) dt
	FROM xyf_ads.ads_user_market_portfolio_label_df
	WHERE pt >= '20241231' 
) base
ON base.user_no = fudai_order.user_no AND fudai_order.订单发起日期 = base.dt
LEFT JOIN first_loan_info   --不是所有复贷用户都能打上虚假给额标签，用的是1月份之后的新客信息
ON first_loan_info.cust_no = fudai_order.cust_no
)


SELECT  loan_orders_info.*
       ,CASE WHEN fy.app_user_id IS NULL THEN '非在会'  ELSE '在会' END                                                                        AS 飞跃在会
       ,CASE WHEN fy.pay_time IS NOT NULL AND DATE(loan_orders_info.first_order_time) >= DATE_ADD(DATE(fy.pay_time),1) THEN 1  ELSE 0 END AS 飞跃在会已扣得
       ,CASE WHEN fx.app_user_id IS NULL THEN '非在会'  ELSE '在会' END                                                                        AS 飞享在会
       ,CASE WHEN fx.pay_time IS NOT NULL AND DATE(loan_orders_info.first_order_time) >= DATE_ADD(DATE(fx.pay_time),1) THEN 1  ELSE 0 END AS 飞享在会已扣得
       ,fy.vip_order_number                                                                                                               AS fy_vip_order_number
       ,fy.start_time                                                                                                                     AS 飞跃会员卡开始时间
       ,fy.end_time                                                                                                                       AS 飞跃会员卡结束时间
       ,fx.vip_order_number                                                                                                               AS fx_vip_order_number
       ,fx.start_time                                                                                                                     AS 飞享会员卡开始时间
       ,fx.end_time                                                                                                                       AS 飞享会员卡结束时间
       ,CASE WHEN kefu.拉黑时间 IS NOT NULL THEN 1  ELSE 0 END                                                                             AS is_拉黑
       ,kefu.拉黑时间                                                                                                                      AS 拉黑时间
       ,kefu.拉黑类型                                                                                                                     
       ,huaxiang.id_number_province                                                                                                       AS 身份证归属省
       ,huaxiang.gender                                                                                                                   AS 性别
       ,CASE WHEN huaxiang.age < 22 THEN '0.~21'
             WHEN huaxiang.age BETWEEN 22 AND 25 THEN '1.22~25'
             WHEN huaxiang.age BETWEEN 26 AND 30 THEN '2.26~30'
             WHEN huaxiang.age BETWEEN 31 AND 35 THEN '3.31~35'
             WHEN huaxiang.age BETWEEN 36 AND 40 THEN '4.36~40'
             WHEN huaxiang.age BETWEEN 41 AND 45 THEN '5.41~45'
             WHEN huaxiang.age BETWEEN 46 AND 50 THEN '6.46~50'
             WHEN huaxiang.age BETWEEN 51 AND 54 THEN '7.51~54'
             WHEN huaxiang.age > 54 THEN '8.55~'  ELSE NULL END                                                                           AS 年龄
       ,CASE WHEN huaxiang.race IS NOT NULL AND huaxiang.race = '汉' THEN '汉族'
             WHEN huaxiang.race IS NOT NULL THEN '其他'  ELSE NULL END                                                                      AS 民族
       ,CASE WHEN huaxiang.education IN ('初中及以下') THEN '1.初中及以下'
             WHEN huaxiang.education IN ('高中/中专/技校','高中/中专技校','高中/中专 /技校','高中/中专／技校') THEN '2.高中/中专/技校'
             WHEN huaxiang.education IN ('专科') THEN '3.专科'
             WHEN huaxiang.education IN ('本科') THEN '4.本科'
             WHEN huaxiang.education IN ('硕士','博士及以上') THEN '5.硕士及以上'  ELSE NULL END                                               AS 自填学历
       ,CASE WHEN huaxiang.monthly_income IN ('1,200~2,500','1,200以内','1200~2500','1200以内') THEN '1.0~2.5k'
             WHEN huaxiang.monthly_income IN ('2,500~4,000','2500~4000') THEN '2.2.5~4k'
             WHEN huaxiang.monthly_income IN ('4,000~6,000','4000~6000') THEN '3.4~6k'
             WHEN huaxiang.monthly_income IN ('6,000~10,000','6000~10000','6000-10000') THEN '4.6~10k'
             WHEN huaxiang.monthly_income IN ('10,000~20,000','10000~20000') THEN '5.10~20k'
             WHEN huaxiang.monthly_income IN ('20,000~50,000','20000~50000') THEN '6.20~50k'
             WHEN huaxiang.monthly_income IN ('50,000以上','50000以上') THEN '7.50k+'  ELSE NULL END                                          AS 自填收入
       ,CASE WHEN huaxiang.code = '00' AND huaxiang.bairong_als_402 IS NULL THEN '1. 0'
             WHEN huaxiang.bairong_als_402 BETWEEN 1 AND 5 THEN '2. 1~5'
             WHEN huaxiang.bairong_als_402 BETWEEN 6 AND 10 THEN '3. 6~10'
             WHEN huaxiang.bairong_als_402 BETWEEN 11 AND 15 THEN '4. 11~15'
             WHEN huaxiang.bairong_als_402 BETWEEN 16 AND 20 THEN '5. 16~20'
             WHEN huaxiang.bairong_als_402 > 20 THEN '6. 20+'  ELSE NULL END                                                            AS 多头
       ,CASE WHEN huaxiang.rh_debt_with_half_houseloan_new <= 0 THEN '1.0'
             WHEN huaxiang.rh_debt_with_half_houseloan_new <= 3000 THEN '2.0-3k'
             WHEN huaxiang.rh_debt_with_half_houseloan_new <= 6000 THEN '3.3k-6k'
             WHEN huaxiang.rh_debt_with_half_houseloan_new <= 8000 THEN '4.6k-8k'
             WHEN huaxiang.rh_debt_with_half_houseloan_new <= 10000 THEN '5.8k-1w'
             WHEN huaxiang.rh_debt_with_half_houseloan_new <= 15000 THEN '6.1w-1.5w'
             WHEN huaxiang.rh_debt_with_half_houseloan_new <= 20000 THEN '7.1.5w-2w'
             WHEN huaxiang.rh_debt_with_half_houseloan_new <= 30000 THEN '8.2w-3w'
             WHEN huaxiang.rh_debt_with_half_houseloan_new > 30000 THEN '9.3w+'  ELSE NULL END                                            AS 负债金额 -- 人行月负债 
       ,CASE WHEN huaxiang.huarong6_318 <= 0 THEN '1.0'
             WHEN huaxiang.huarong6_318 <= 3000 THEN '2.0-3k'
             WHEN huaxiang.huarong6_318 <= 6000 THEN '3.3k-6k'
             WHEN huaxiang.huarong6_318 <= 8000 THEN '4.6k-8k'
             WHEN huaxiang.huarong6_318 <= 10000 THEN '5.8k-1w'
             WHEN huaxiang.huarong6_318 <= 15000 THEN '6.1w-1.5w'
             WHEN huaxiang.huarong6_318 <= 20000 THEN '7.1.5w-2w'
             WHEN huaxiang.huarong6_318 <= 30000 THEN '8.2w-3w'
             WHEN huaxiang.huarong6_318 > 30000 THEN '9.3w+'  ELSE NULL END                                                               AS 信用卡平均授信金额
	   ,tousu.*EXCEPT(order_number)
FROM loan_orders_info
-- 风险通过时间 
LEFT JOIN
(
	SELECT  DISTINCT ori_order_number
	       ,risk_success_time --去重，只留一条包含风险通过时间的记录 
	FROM xyf_dwd.dwd_inloan_loan_apply_hf
	WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_loan_apply_hf')
	AND DATE(main_date_created) >= '2025-01-01' -- 预过滤 
 
) r
ON loan_orders_info.first_order_number = r.ori_order_number
-- 飞跃在会状态 （新逻辑：基于会员卡有效期） 
LEFT JOIN
(
	SELECT  app_user_id
	       ,vip_order_number
	       ,order_time
	       ,pay_time
	       ,start_time
	       ,coalesce(CAST(failure_time AS DATETIME),end_time) AS end_time --飞跃合约期外退款，failure_time = end_time 
	       ,loan_order_number
	       ,vip_status
	       ,renew_period
	FROM xyf_dwd.dwd_inloan_leap_vip_order_hf
	WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_leap_vip_order_hf')
	AND DATE(order_time) >= '2025-01-01' 
) fy
ON loan_orders_info.user_no = fy.app_user_id AND r.risk_success_time >= fy.start_time AND r.risk_success_time <= fy.end_time -- 会员卡开始时间 <= 风险通过时间 <= 会员卡结束时间
-- 飞享在会状态 
LEFT JOIN
(
	SELECT  app_user_id
	       ,vip_order_number
	       ,first_vip_order_number
	       ,order_time
	       ,pay_time
	       ,start_time
	       ,LEAST(CAST(failure_time AS DATETIME),end_time) AS end_time --飞享存在合约期外退款，把refund_time作为failure_time，因此取两者较小值 
	       ,order_number_loan                              AS loan_order_number
	FROM xyf_dwd.dwd_user_vip_order_df
	WHERE pt = MAX_PT('xyf_dwd.dwd_user_vip_order_df')
	AND vip_card_type = 1
	-- AND if_validation <> 0  用户发起成功后投诉退款标记为0，此处为了看用户是否办卡，剔除筛选条件
	AND DATE(order_time) >= '2025-01-01' 
) fx
ON loan_orders_info.user_no = fx.app_user_id AND r.risk_success_time >= fx.start_time AND r.risk_success_time <= fx.end_time
-- 客服高危用户拉黑状态
LEFT JOIN
(
	SELECT  DATE(t1.created_time) AS 拉黑时间
	       ,CASE WHEN t1.list_type = 'ip' THEN 'ip封禁'
		         WHEN t1.list_type = 'device_id' THEN '设备号封禁'
		         WHEN t1.list_type = 'open_id' THEN '微信号封禁' ELSE '其他' END     AS 拉黑类型
	       ,t2.app_user_id
	FROM xyf_dwd.dwd_listcore_user_list_df_v t1
	INNER JOIN
	(
		SELECT  created_ip AS match_field --用户创建ip 
		       ,app_user_id
		       ,'ip'       AS match_type -- 标记关联类型为ip 
		FROM xyf_dim.dim_user_app_basic_info_df
		WHERE pt = MAX_PT('xyf_dim.dim_user_app_basic_info_df')
		AND created_ip IS NOT NULL -- 过滤空值，提升效率 
 
		UNION ALL
		SELECT  open_id   AS match_field --小程序用户标识符，唯一标识用户 
		       ,app_user_id
		       ,'open_id' AS match_type -- 标记关联类型为open_id 
		FROM xyf_dim.dim_user_app_basic_info_df
		WHERE pt = MAX_PT('xyf_dim.dim_user_app_basic_info_df')
		AND open_id IS NOT NULL 

		UNION ALL
		SELECT  last_login_device_id AS match_field -- 最后一次登录设备号 
		       ,app_user_id
		       ,'device_id'          AS match_type -- 标记关联类型为device_id 
		FROM xyf_dim.dim_user_app_basic_info_df
		WHERE pt = MAX_PT('xyf_dim.dim_user_app_basic_info_df')
		AND last_login_device_id IS NOT NULL 
	) t2
	ON t1.list_detail = t2.match_field AND t1.list_type = t2.match_type
	WHERE t1.list_type IN ('ip', 'open_id', 'device_id')
	AND t1.status = 2 
	QUALIFY ROW_NUMBER() OVER ( PARTITION BY t2.app_user_id ORDER BY t1.created_time DESC ) = 1 
) kefu --- 客服这里有重复 
ON loan_orders_info.user_no = kefu.app_user_id
LEFT JOIN xyf_bi.fk_risk_order_feature_df_forbi huaxiang
ON loan_orders_info.first_order_number = huaxiang.ori_order_number AND huaxiang.pt = MAX_PT('xyf_bi.fk_risk_order_feature_df_forbi')
-- 投诉情况（一个用户可能有多条投诉记录）
LEFT JOIN
(
	SELECT  order_number -- 订单号 
	       ,channel_1_type                                                                                 AS 投诉渠道
	       ,channel_2_type                                                                                 AS 投诉渠道细分
	       ,CASE WHEN channel_1_type LIKE '%资方%' AND channel_2_type LIKE '%监管%' THEN '重渠-资方监管' --资方监管 
	             WHEN channel_1_type LIKE '%监管%' THEN '重渠-属地监管' --属地监管 
	             WHEN channel_1_type LIKE '%公安%' THEN '重渠-公安'
	             WHEN channel_1_type LIKE '%人民银行%' THEN '重渠-人行'
	             WHEN channel_1_type = '互联网平台' THEN '黑猫'
	             WHEN channel_1_type = '资方' AND channel_2_type not LIKE '%监管%' THEN '资方非监管'  ELSE '其他' END AS 投诉渠道标签
	       ,task_type_name                                                                                 AS 投诉大类
	       ,question_type_name                                                                             AS 投诉小类
	       ,comment                                                                                        AS 投诉内容
	       ,app_user_id
	       ,create_time                                                                                    AS 投诉时间
	       ,date(create_time)                                                                              AS 投诉日期
	FROM xyf_ads.bi_customer_service_ts_task_df
	WHERE pt = max_pt("xyf_ads.bi_customer_service_ts_task_df") 
) tousu
ON loan_orders_info.order_number = tousu.order_number AND DATEDIFF(tousu.投诉日期, loan_orders_info.放款日期) BETWEEN 0 AND 30


In [1]:
import os
from odps import ODPS
import numpy as np
import pandas as pd

##initialize odps
o = ODPS(
    # （推荐）确保已设置环境变量。
    # 确保ALIBABA_CLOUD_ACCESS_KEY_ID环境变量设置为用户 Access Key ID。
    access_id=os.getenv('ALIBABA_CLOUD_ACCESS_KEY_ID'),
    
    # 确保ALIBABA_CLOUD_ACCESS_KEY_SECRET环境变量设置为用户Access Key Secret。
    secret_access_key=os.getenv('ALIBABA_CLOUD_ACCESS_KEY_SECRET'),
    project='xyf_jingying_dev',
    endpoint='https://service.cn-beijing.maxcompute.aliyun.com/api',
)

import shutil
from openpyxl import load_workbook
from openpyxl.styles import Font, Fill, Border, Alignment, Side, PatternFill
from win32com.client import Dispatch
from datetime import datetime , timedelta 

def write_dataframe_to_excel_com(file_path, dataframes_dict, start_row=1, start_col=1, include_header=True):
    """
    Args:
        file_path: Excel文件路径
        dataframes_dict: {sheet_name: dataframe} 字典
        start_row: 从第几行开始写入（默认1）
        start_col: 从第几列开始写入（默认1）
        include_header: 是否写入表头（默认True）
    """
    # 启动Excel应用
    excel_app = Dispatch("Excel.Application")
    excel_app.Visible = False               # 后台运行
    excel_app.DisplayAlerts = False         # 禁用警告对话框

    try:
        # 打开工作簿
        workbook = excel_app.Workbooks.Open(file_path)

        for sheet_name, df in dataframes_dict.items():
            try:
                # 获取工作表
                worksheet = workbook.Worksheets(sheet_name)

                # 连表头/旧内容一起清空
                used = worksheet.UsedRange
                if used is not None and used.Rows.Count > 0 and used.Columns.Count > 0:
                    used.ClearContents()

                if df is not None and not df.empty:
                    # Excel/COM 识别 None（写成空单元格），不识别 Pandas 的 pd.NA，写入前：把 Pandas 的 NA/NaN/inf 转成 Excel 可接受的 None
                    df_to_write = df.copy()
                    df_to_write = df_to_write.replace([np.inf, -np.inf], np.nan)
                    df_to_write = df_to_write.astype('object')       # 防止 NAType 保留在扩展dtype里
                    df_to_write = df_to_write.where(pd.notna(df_to_write), None)

                    # 要写入的数据：可选表头 + 数据
                    if include_header:
                        values = [df_to_write.columns.tolist()] + df_to_write.values.tolist()
                    else:
                        values = df_to_write.values.tolist()

                    n_rows = len(values)
                    n_cols = len(values[0]) if n_rows > 0 else 0

                    end_row = start_row + n_rows - 1
                    end_col = start_col + n_cols - 1

                    write_range = worksheet.Range(
                        worksheet.Cells(start_row, start_col),
                        worksheet.Cells(end_row, end_col),
                    )
                    write_range.Value = values

                print(f"成功写入工作表: {sheet_name}")

            except Exception as e:
                print(f"写入工作表 {sheet_name} 时出错: {e}")
                continue
        
        workbook.RefreshAll()                       # 刷新所有数据连接和透视表
        excel_app.Calculate()                       # 强制重新计算公式
        excel_app.CalculateUntilAsyncQueriesDone()  # 等待所有操作完成
        
        # 保存并关闭
        workbook.Save()
        workbook.Close()
        print(f"文件已保存: {file_path}")

    except Exception as e:
        print(f"操作Excel文件时出错: {e}")

    finally:
        excel_app.Quit()        # 退出Excel应用


def add_pivot_calculated_fields_com(
    file_path: str,
    sheet_name: str,
    pivot_name: str | None = None,
    pivot_index: int = 1,
    fields: dict | None = None,
    refresh: bool = True,
    add_to_values: bool = True,          # 是否自动加入“值”区域
    skip_if_exists: bool = True,         # 同名计算字段已存在时：True=跳过；False=尝试删除后重建（不稳定）
    verbose: bool = True,                # 是否打印过程日志
):
    """
    在已有 PivotTable 上批量添加 Calculated Fields（计算字段）
    - 可选：自动加入值区域
    - 若遇到同名计算字段/异常：记录并跳过，继续执行剩余字段
    - 执行结束：print 未成功字段清单

    fields 格式：
    {
      "金额加权平均定价2": {"formula": "='金额*定价(%)'/放款金额", "number_format": "0.00", "caption": "金额加权平均定价2"},
      ...
    }
    """
    if not fields:
        raise ValueError("fields 不能为空，例如：{'字段名': {'formula': '=<expr>', 'number_format': '0.00%'}}")

    failed = []  

    excel_app = Dispatch("Excel.Application")
    excel_app.Visible = False
    excel_app.DisplayAlerts = False

    # 常量
    xlDataField = 4  # PivotField 作为“值”区域

    try:
        wb = excel_app.Workbooks.Open(file_path)
        ws = wb.Worksheets(sheet_name)

        # 选取 PivotTable（建议用 pivot_name）
        pvt = ws.PivotTables(pivot_name) if pivot_name else ws.PivotTables(pivot_index)

        if verbose:
            try:
                print(f"[Pivot] sheet={sheet_name} pivot={pvt.Name} cache_index={pvt.PivotCache().Index}")
            except Exception:
                print(f"[Pivot] sheet={sheet_name}")

        for field_name, cfg in fields.items():
            cfg = cfg or {}
            formula = cfg.get("formula")
            if not formula:
                failed.append({"field": field_name, "reason": "missing_formula", "detail": "cfg 中缺少 formula"})
                continue

            number_format = cfg.get("number_format", "General")
            caption = cfg.get("caption", field_name)

            # 1) 检查同名计算字段是否已存在
            exists = False
            try:
                _ = pvt.CalculatedFields(field_name)  # 存在则不报错
                exists = True
            except Exception:
                exists = False

            if exists and skip_if_exists:
                failed.append({"field": field_name, "reason": "already_exists", "detail": "计算字段已存在，按配置跳过"})
                if verbose:
                    print(f"[Skip] {field_name} 已存在，跳过")
                continue

            # 2) （可选）尝试删除旧计算字段（注意：删除常因被其他透视表引用而失败）
            if exists and not skip_if_exists:
                try:
                    pvt.CalculatedFields(field_name).Delete()
                except Exception as e:
                    failed.append({"field": field_name, "reason": "delete_failed", "detail": str(e)})
                    if verbose:
                        print(f"[Fail] 删除旧计算字段失败: {field_name} | {e}")
                    continue

            # 3) 新增计算字段
            try:
                pvt.CalculatedFields().Add(Name=field_name, Formula=formula)
                if verbose:
                    print(f"[OK] Add CalculatedField: {field_name} | {formula}")
            except Exception as e:
                failed.append({"field": field_name, "reason": "add_failed", "detail": str(e)})
                if verbose:
                    print(f"[Fail] Add CalculatedField 失败: {field_name} | {e}")
                continue

            # 4) 是否加入“值区域”
            if add_to_values:
                try:
                    # 更直接：把 PivotField 放到值区域（兼容性更高）
                    pf = pvt.PivotFields(field_name)
                    pf.Orientation = xlDataField
                    pf.NumberFormat = number_format
                    # 如果你想控制显示名，可在 Excel 里再改；COM 里改 Name 有时会触发异常
                    if verbose:
                        print(f"[OK] Add to Values: {field_name} | format={number_format}")
                except Exception as e:
                    # 加值区域失败不影响“计算字段已创建”，但按你的要求也算未成功设置
                    failed.append({"field": field_name, "reason": "add_to_values_failed", "detail": str(e)})
                    if verbose:
                        print(f"[Fail] 加入值区域失败: {field_name} | {e}")
                    continue

        # 5) 刷新（尽量只刷新当前透视表，避免 RefreshAll 造成锁/异步）
        if refresh:
            try:
                pvt.RefreshTable()
            except Exception:
                try:
                    wb.RefreshAll()
                    excel_app.Calculate()
                    try:
                        excel_app.CalculateUntilAsyncQueriesDone()
                    except Exception:
                        pass
                except Exception:
                    pass

        wb.Save()
        wb.Close()

    finally:
        excel_app.Quit()

    # 汇总输出
    if failed:
        print("\n===== 未成功设置的计算字段（含跳过项）=====")
        for item in failed:
            print(f"- {item.get('field')} | {item.get('reason')} | {item.get('detail')}")
        print("===== 结束 =====\n")
    else:
        print("\n全部计算字段设置成功。\n")

    return failed


#### 投诉概览

In [6]:
## 投诉概览
query='''
SELECT  放款月
       ,asset_type_flag
       ,period
       ,is_虚假给额
       ,loan_flag
       ,飞跃在会
       ,飞享在会
       ,授信额度区间
       ,投诉渠道
       ,投诉渠道标签
       ,投诉大类
       ,COUNT(DISTINCT user_no)                                              AS 放款人数
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(投诉日期,放款日期) = 0 THEN user_no end)   AS 放款T0投诉
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(投诉日期,放款日期) <= 3 THEN user_no end)  AS 放款T3投诉
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(投诉日期,放款日期) <= 7 THEN user_no end)  AS 放款T7投诉
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(投诉日期,放款日期) <= 15 THEN user_no end) AS 放款T15投诉
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(投诉日期,放款日期) <= 30 THEN user_no end)  AS 放款T30投诉
FROM lss_order_loan_tousu
GROUP BY  放款月
         ,asset_type_flag
         ,period
         ,is_虚假给额
         ,loan_flag
         ,飞跃在会
         ,飞享在会
         ,授信额度区间
         ,投诉渠道
         ,投诉渠道标签
         ,投诉大类

'''
#底表数据
result = o.execute_sql(query)
tousu_stats = result.open_reader().to_pandas()

# 字符串字段
str_cols = [
          '放款月'
         ,'asset_type_flag'
         ,'period'
         ,'授信渠道'
         ,'is_虚假给额'
         ,'飞跃在会'
         ,'飞享在会'
         ,'授信额度区间'
         ,'投诉渠道'
         ,'投诉渠道标签'
         ,'投诉大类'
         ,'身份证归属省'
         ,'年龄'
         ,'民族'
         ,'自填学历'
         ,'自填收入'
         ,'多头'
         ,'负债金额'
         ,'信用卡平均授信金额'
]
for col in str_cols:
    if col in tousu_stats.columns:
        # tousu_stats[col] = tousu_stats[col].astype('string').fillna('')  # 填充空值为空字符串
        tousu_stats[col] = tousu_stats[col].astype('string')

# 整数字段
int_cols = [
        '放款人数', 
        '放款T0投诉'
       ,'放款T3投诉'
       ,'放款T7投诉'
       ,'放款T15投诉'
       ,'放款T30投诉'
]
for col in int_cols:
    if col in tousu_stats.columns:
        tousu_stats[col] = pd.to_numeric(tousu_stats[col], errors='coerce').astype('Int64')  # 使用Pandas 的 Int64（可空整数）/ NumPy 的 int64（普通整数）

file_path = r"D:\9.极限给额专题分析\投诉专项分析.xlsx"
write_dataframe_to_excel_com(file_path, {"投诉概览": tousu_stats}, start_row=1, include_header=True)

成功写入工作表: 投诉概览
文件已保存: D:\9.极限给额专题分析\投诉专项分析.xlsx


#### 投诉细分

In [7]:
## 投诉细分
query='''
SELECT  放款月
       ,is_虚假给额
       ,loan_flag
       ,飞跃在会
       ,飞享在会
       ,授信额度区间
       ,投诉渠道标签
       ,投诉大类
       ,投诉小类
       ,COUNT(DISTINCT user_no) AS 投诉人数
FROM
(
	SELECT  *
	FROM lss_order_loan_tousu
	WHERE 投诉渠道标签 IS NOT NULL 
       QUALIFY ROW_NUMBER() OVER (PARTITION BY user_no, 投诉小类 ORDER BY 投诉时间 ASC) = 1 
)
GROUP BY  放款月
         ,is_虚假给额
         ,loan_flag
         ,飞跃在会
         ,飞享在会
         ,授信额度区间
         ,投诉渠道标签
         ,投诉大类
         ,投诉小类
'''
#底表数据
result = o.execute_sql(query)
tousu_stats = result.open_reader().to_pandas()

# 字符串字段
str_cols = [
          '放款月'
         ,'asset_type_flag'
         ,'period'
         ,'授信渠道'
         ,'is_虚假给额'
         ,'飞跃在会'
         ,'飞享在会'
         ,'授信金额区间'
         ,'投诉渠道'
         ,'投诉渠道标签'
         ,'投诉大类'
         ,'投诉小类'
         ,'身份证归属省'
         ,'年龄'
         ,'民族'
         ,'自填学历'
         ,'自填收入'
         ,'多头'
         ,'负债金额'
         ,'信用卡平均授信金额'
]
for col in str_cols:
    if col in tousu_stats.columns:
        # tousu_stats[col] = tousu_stats[col].astype('string').fillna('')  # 填充空值为空字符串
        tousu_stats[col] = tousu_stats[col].astype('string')

# 整数字段
int_cols = ['投诉人数']
for col in int_cols:
    if col in tousu_stats.columns:
        tousu_stats[col] = pd.to_numeric(tousu_stats[col], errors='coerce').astype('Int64')  # 使用Pandas 的 Int64（可空整数）/ NumPy 的 int64（普通整数）

file_path = r"D:\9.极限给额专题分析\投诉专项分析.xlsx"
write_dataframe_to_excel_com(file_path, {"投诉细分": tousu_stats}, start_row=1, include_header=True)

成功写入工作表: 投诉细分
文件已保存: D:\9.极限给额专题分析\投诉专项分析.xlsx


#### 会员卡

In [10]:
## 会员卡
query='''
SELECT  放款月
       ,asset_type_flag
       ,period
       ,is_虚假给额
       ,loan_flag
       ,飞跃在会
       ,飞享在会
       ,授信额度区间
       ,投诉渠道标签
       ,投诉大类
       ,COUNT(DISTINCT user_no)                                                   AS 放款人数
       ,COUNT(DISTINCT CASE WHEN 飞跃在会 = '在会' THEN user_no end)               AS 飞跃在会人数
       ,COUNT(DISTINCT CASE WHEN 飞享在会 = '在会' THEN user_no end)               AS 飞享在会人数
       ,COUNT(DISTINCT CASE WHEN 投诉渠道标签 IN ( '重渠-资方监管','重渠-属地监管','重渠-公安','重渠-人行') THEN user_no end)    AS 重渠投诉人数
       ,COUNT(DISTINCT CASE WHEN 投诉渠道标签 IS NOT NULL  THEN user_no end)    AS 投诉人数
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(投诉日期,放款日期) = 0 THEN user_no end)   AS 放款T0投诉
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(投诉日期,放款日期) <= 3 THEN user_no end)  AS 放款T3投诉
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(投诉日期,放款日期) <= 7 THEN user_no end)  AS 放款T7投诉
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(投诉日期,放款日期) <= 15 THEN user_no end) AS 放款T15投诉
       ,COUNT(DISTINCT CASE WHEN DATEDIFF(投诉日期,放款日期) <= 30 THEN user_no end) AS 放款T30投诉
       
FROM lss_order_loan_tousu
GROUP BY  放款月
         ,asset_type_flag
         ,period
         ,is_虚假给额
         ,loan_flag
         ,飞跃在会
         ,飞享在会
         ,授信额度区间
         ,投诉渠道标签
         ,投诉大类
'''
#底表数据
result = o.execute_sql(query)
tousu_stats = result.open_reader().to_pandas()

# 字符串字段
str_cols = [
          '放款月'
         ,'asset_type_flag'
         ,'period'
         ,'授信渠道'
         ,'is_虚假给额'
         ,'飞跃在会'
         ,'飞享在会'
         ,'授信金额区间'
         ,'投诉渠道'
         ,'投诉渠道标签'
         ,'投诉大类'
         ,'投诉小类'
]
for col in str_cols:
    if col in tousu_stats.columns:
        # tousu_stats[col] = tousu_stats[col].astype('string').fillna('')  # 填充空值为空字符串
        tousu_stats[col] = tousu_stats[col].astype('string')

# 整数字段
int_cols = [ '放款人数', '飞跃在会人数', '飞享在会人数', '投诉人数', '重渠投诉人数' ,'放款T0投诉' ,'放款T3投诉','放款T7投诉','放款T15投诉' ,'放款T30投诉']
for col in int_cols:
    if col in tousu_stats.columns:
        tousu_stats[col] = pd.to_numeric(tousu_stats[col], errors='coerce').astype('Int64')  # 使用Pandas 的 Int64（可空整数）/ NumPy 的 int64（普通整数）

file_path = r"D:\9.极限给额专题分析\投诉专项分析.xlsx"
write_dataframe_to_excel_com(file_path, {"会员卡": tousu_stats}, start_row=1, include_header=True)

成功写入工作表: 会员卡
文件已保存: D:\9.极限给额专题分析\投诉专项分析.xlsx


In [11]:
calc_fields = { 
    '飞跃在会率': {"formula": "='飞跃在会人数'/放款人数", "number_format": "0.00%"},
    '飞享在会率': {"formula": "='飞享在会人数'/放款人数", "number_format": "0.00%"},
    '重渠投诉率': {"formula": "='重渠投诉人数'/放款人数", "number_format": "0.00%"},
    '投诉率': {"formula": "='投诉人数'/放款人数", "number_format": "0.00%"},
    "T0投诉率": {"formula": "='放款t0投诉'/放款人数", "number_format": "0.00%"},
    "T3投诉率": {"formula": "='放款t3投诉'/放款人数", "number_format": "0.00%"},
    "T7投诉率": {"formula": "='放款t7投诉'/放款人数", "number_format": "0.00%"},
    "T15投诉率": {"formula": "='放款t15投诉'/放款人数", "number_format": "0.00%"},
    "T30投诉率": {"formula": "='放款t30投诉'/放款人数", "number_format": "0.00%"},
}

failed = add_pivot_calculated_fields_com(
    file_path=file_path,
    sheet_name="by会员卡",
    pivot_name="数据透视表2",
    fields=calc_fields,
    refresh=True,
    add_to_values=True,      # 自动加入值区域
    skip_if_exists=True,     # 同名就记录并跳过（避免第二次 Add “发生意外”）
    verbose=True,
) 

[Pivot] sheet=by会员卡 pivot=数据透视表2 cache_index=2
[Fail] Add CalculatedField 失败: 飞跃在会率 | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
[Fail] Add CalculatedField 失败: 飞享在会率 | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
[Fail] Add CalculatedField 失败: 重渠投诉率 | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
[OK] Add CalculatedField: 投诉率 | ='投诉人数'/放款人数
[OK] Add to Values: 投诉率 | format=0.00%
[Fail] Add CalculatedField 失败: T0投诉率 | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
[Fail] Add CalculatedField 失败: T3投诉率 | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
[Fail] Add CalculatedField 失败: T7投诉率 | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
[Fail] Add CalculatedField 失败: T15投诉率 | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
[Fail] Add CalculatedField 失败: T30投诉率 | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)

===== 未成功设

#### 用户画像

In [19]:
## 用户画像
query='''
SELECT  放款月
       ,loan_flag
       ,授信额度区间
       -- ,身份证归属省
       ,年龄
       ,民族
       ,自填学历
       --,自填收入
       --,多头
       --,负债金额
       --,信用卡平均授信金额
       ,COUNT(DISTINCT user_no)                                                                     AS 放款人数
       ,COUNT(DISTINCT CASE WHEN 飞跃在会 = '在会' THEN user_no end)                                      AS 飞跃在会人数
       ,COUNT(DISTINCT CASE WHEN 飞享在会 = '在会' THEN user_no end)                                      AS 飞享在会人数
       ,COUNT(DISTINCT CASE WHEN 投诉渠道标签 IN ( '重渠-资方监管','重渠-属地监管','重渠-公安','重渠-人行') THEN user_no end) AS 重渠投诉人数
       ,COUNT(DISTINCT CASE WHEN 投诉渠道标签 IS NOT NULL THEN user_no end)                               AS 投诉人数
FROM lss_order_loan_tousu
GROUP BY  放款月
         ,loan_flag
         ,授信额度区间
         -- ,身份证归属省
         ,年龄
         ,民族
         ,自填学历
         --,自填收入
         --,多头
         --,负债金额
         --,信用卡平均授信金额
'''
#底表数据
result = o.execute_sql(query)
tousu_stats = result.open_reader().to_pandas()

# 字符串字段
str_cols = [
          '放款月'
         ,'asset_type_flag'
         ,'period'
         ,'授信渠道'
         ,'is_虚假给额'
         ,'飞跃在会'
         ,'飞享在会'
         ,'授信金额区间'
         ,'投诉渠道'
         ,'投诉渠道标签'
         ,'投诉大类'
         ,'投诉小类'
         ,'身份证归属省'
         ,'年龄'
         ,'民族'
         ,'自填学历'
         ,'自填收入'
         ,'多头'
         ,'负债金额'
         ,'信用卡平均授信金额'
]
for col in str_cols:
    if col in tousu_stats.columns:
        # tousu_stats[col] = tousu_stats[col].astype('string').fillna('')  # 填充空值为空字符串
        tousu_stats[col] = tousu_stats[col].astype('string')

# 整数字段
int_cols = [ '放款人数', '飞跃在会人数', '飞享在会人数', '投诉人数', '重渠投诉人数' ]
for col in int_cols:
    if col in tousu_stats.columns:
        tousu_stats[col] = pd.to_numeric(tousu_stats[col], errors='coerce').astype('Int64')  # 使用Pandas 的 Int64（可空整数）/ NumPy 的 int64（普通整数）

file_path = r"D:\9.极限给额专题分析\投诉专项分析.xlsx"
write_dataframe_to_excel_com(file_path, {"用户画像": tousu_stats}, start_row=1, include_header=True)

成功写入工作表: 用户画像
文件已保存: D:\9.极限给额专题分析\投诉专项分析.xlsx


In [ ]:
calc_fields = { 
    '飞跃在会率': {"formula": "='飞跃在会人数'/放款人数", "number_format": "0.00%"},
    '飞享在会率': {"formula": "='飞享在会人数'/放款人数", "number_format": "0.00%"},
    '拉黑率': {"formula": "='拉黑人数'/放款人数", "number_format": "0.00%"},
    '重渠投诉率': {"formula": "='重渠投诉人数'/放款人数", "number_format": "0.00%"},
    '投诉率': {"formula": "='投诉人数'/放款人数", "number_format": "0.00%"},
}

failed = add_pivot_calculated_fields_com(
    file_path=file_path,
    sheet_name="客户拉黑标签",
    pivot_name="数据透视表1",
    fields=calc_fields,
    refresh=True,
    add_to_values=True,      # 自动加入值区域
    skip_if_exists=True,     # 同名就记录并跳过（避免第二次 Add “发生意外”）
    verbose=True,
) 

In [20]:
## 用户画像1
query='''
SELECT  放款月 
        -- ,asset_type_flag
    --    ,period
       ,is_虚假给额
       ,loan_flag
       ,授信额度区间
       ,身份证归属省
    --    ,年龄
    --    ,民族
    --    ,自填学历
       --,自填收入
    --    ,多头
       --,负债金额
       --,信用卡平均授信金额
       ,COUNT(DISTINCT user_no)                                                                     AS 放款人数
       ,COUNT(DISTINCT CASE WHEN 飞跃在会 = '在会' THEN user_no end)                                      AS 飞跃在会人数
       ,COUNT(DISTINCT CASE WHEN 飞享在会 = '在会' THEN user_no end)                                      AS 飞享在会人数
       ,COUNT(DISTINCT CASE WHEN 投诉渠道标签 IN ( '重渠-资方监管','重渠-属地监管','重渠-公安','重渠-人行') THEN user_no end) AS 重渠投诉人数
       ,COUNT(DISTINCT CASE WHEN 投诉渠道标签 IS NOT NULL THEN user_no end)                               AS 投诉人数
FROM lss_order_loan_tousu
GROUP BY  放款月 
        -- ,asset_type_flag
    --    ,period
       ,is_虚假给额
       ,loan_flag
         ,授信额度区间
         ,身份证归属省
'''
#底表数据
result = o.execute_sql(query)
tousu_stats = result.open_reader().to_pandas()

# 字符串字段
str_cols = [
          '放款月'
         ,'asset_type_flag'
         ,'period'
         ,'授信渠道'
         ,'is_虚假给额'
         ,'飞跃在会'
         ,'飞享在会'
         ,'授信金额区间'
         ,'投诉渠道'
         ,'投诉渠道标签'
         ,'投诉大类'
         ,'投诉小类'
         ,'身份证归属省'
         ,'年龄'
         ,'民族'
         ,'自填学历'
         ,'自填收入'
         ,'多头'
         ,'负债金额'
         ,'信用卡平均授信金额'
]
for col in str_cols:
    if col in tousu_stats.columns:
        # tousu_stats[col] = tousu_stats[col].astype('string').fillna('')  # 填充空值为空字符串
        tousu_stats[col] = tousu_stats[col].astype('string')

# 整数字段
int_cols = [ '放款人数', '飞跃在会人数', '飞享在会人数', '投诉人数', '重渠投诉人数' ]
for col in int_cols:
    if col in tousu_stats.columns:
        tousu_stats[col] = pd.to_numeric(tousu_stats[col], errors='coerce').astype('Int64')  # 使用Pandas 的 Int64（可空整数）/ NumPy 的 int64（普通整数）

file_path = r"D:\9.极限给额专题分析\投诉专项分析.xlsx"
write_dataframe_to_excel_com(file_path, {"用户画像by省份": tousu_stats}, start_row=1, include_header=True)

成功写入工作表: 用户画像by省份
文件已保存: D:\9.极限给额专题分析\投诉专项分析.xlsx


#### 客服拉黑

In [13]:
## 客服拉黑
query='''
SELECT  
        放款月
       ,asset_type_flag
       ,period
       ,is_虚假给额
       ,loan_flag
       ,飞跃在会
       ,飞享在会
       ,授信额度区间
       ,投诉渠道标签
       ,投诉大类
       ,is_拉黑
       ,拉黑类型
       ,COUNT(DISTINCT user_no)                                                                     AS 放款人数
       ,COUNT(DISTINCT CASE WHEN is_拉黑 = 1 THEN user_no end)                                      AS 拉黑人数
       ,COUNT(DISTINCT CASE WHEN 飞跃在会 = '在会' THEN user_no end)                                      AS 飞跃在会人数
       ,COUNT(DISTINCT CASE WHEN 飞享在会 = '在会' THEN user_no end)                                      AS 飞享在会人数
       ,COUNT(DISTINCT CASE WHEN 投诉渠道标签 IN ( '重渠-资方监管','重渠-属地监管','重渠-公安','重渠-人行') THEN user_no end) AS 重渠投诉人数
       ,COUNT(DISTINCT CASE WHEN 投诉渠道标签 IS NOT NULL THEN user_no end)                               AS 投诉人数
FROM lss_order_loan_tousu
GROUP BY  放款月
       ,asset_type_flag
       ,period
       ,is_虚假给额
       ,loan_flag
       ,飞跃在会
       ,飞享在会
       ,授信额度区间
       ,投诉渠道标签
       ,投诉大类
       ,is_拉黑
       ,拉黑类型
'''
#底表数据
result = o.execute_sql(query)
tousu_stats = result.open_reader().to_pandas()

# 字符串字段
str_cols = [
          '放款月'
         ,'asset_type_flag'
         ,'period'
         ,'授信渠道'
         ,'is_虚假给额'
         ,'飞跃在会'
         ,'飞享在会'
         ,'授信金额区间'
         ,'投诉渠道'
         ,'投诉渠道标签'
         ,'投诉大类'
         ,'拉黑类型'
         ,'is_拉黑'
]
for col in str_cols:
    if col in tousu_stats.columns:
        # tousu_stats[col] = tousu_stats[col].astype('string').fillna('')  # 填充空值为空字符串
        tousu_stats[col] = tousu_stats[col].astype('string')

# 整数字段
int_cols = [ '放款人数', '飞跃在会人数', '飞享在会人数', '投诉人数', '重渠投诉人数' ,'拉黑人数']
for col in int_cols:
    if col in tousu_stats.columns:
        tousu_stats[col] = pd.to_numeric(tousu_stats[col], errors='coerce').astype('Int64')  # 使用Pandas 的 Int64（可空整数）/ NumPy 的 int64（普通整数）

file_path = r"D:\9.极限给额专题分析\投诉专项分析.xlsx"
write_dataframe_to_excel_com(file_path, {"客服拉黑": tousu_stats}, start_row=1, include_header=True)

成功写入工作表: 客服拉黑
文件已保存: D:\9.极限给额专题分析\投诉专项分析.xlsx


In [14]:
calc_fields = { 
    '飞跃在会率': {"formula": "='飞跃在会人数'/放款人数", "number_format": "0.00%"},
    '飞享在会率': {"formula": "='飞享在会人数'/放款人数", "number_format": "0.00%"},
    '拉黑率': {"formula": "='拉黑人数'/放款人数", "number_format": "0.00%"},
    '重渠投诉率': {"formula": "='重渠投诉人数'/放款人数", "number_format": "0.00%"},
    '投诉率': {"formula": "='投诉人数'/放款人数", "number_format": "0.00%"},
}

failed = add_pivot_calculated_fields_com(
    file_path=file_path,
    sheet_name="客户拉黑标签",
    pivot_name="数据透视表1",
    fields=calc_fields,
    refresh=True,
    add_to_values=True,      # 自动加入值区域
    skip_if_exists=True,     # 同名就记录并跳过（避免第二次 Add “发生意外”）
    verbose=True,
) 

[Pivot] sheet=客户拉黑标签 pivot=数据透视表1 cache_index=2
[Fail] Add CalculatedField 失败: 飞跃在会率 | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
[Fail] Add CalculatedField 失败: 飞享在会率 | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
[OK] Add CalculatedField: 拉黑率 | ='拉黑人数'/放款人数
[OK] Add to Values: 拉黑率 | format=0.00%
[Fail] Add CalculatedField 失败: 重渠投诉率 | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
[Fail] Add CalculatedField 失败: 投诉率 | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)

===== 未成功设置的计算字段（含跳过项）=====
- 飞跃在会率 | add_failed | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
- 飞享在会率 | add_failed | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
- 重渠投诉率 | add_failed | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
- 投诉率 | add_failed | (-2147352567, '发生意外。', (0, None, None, None, 0, -2146827284), None)
===== 结束 =====



#### 重渠投诉

In [18]:
## 客服拉黑
query='''
SELECT  放款月
       ,is_虚假给额
       ,loan_flag
       ,飞跃在会
       ,飞享在会
       ,is_拉黑
       ,拉黑类型
       ,投诉日期
       ,投诉渠道细分
       ,投诉渠道标签
       ,投诉大类
       ,投诉小类
       ,投诉内容
FROM lss_order_loan_tousu 
-- WHERE loan_flag = "首贷"
-- AND is_虚假给额 <> '非虚假给额' 
WHERE 投诉渠道标签 IN ( '重渠-资方监管', '重渠-属地监管', '重渠-公安', '重渠-人行') 
'''
#底表数据
result = o.execute_sql(query)
tousu_stats = result.open_reader().to_pandas()

# 字符串字段
str_cols = [
          '放款月'
         ,'asset_type_flag'
         ,'period'
         ,'授信渠道'
         ,'is_虚假给额'
         ,'飞跃在会'
         ,'飞享在会'
         ,'投诉渠道细分'
         ,'投诉渠道标签'
         ,'投诉大类'
         ,'投诉小类'
         ,'投诉内容'
         ,'拉黑类型'
         ,'is_拉黑'
]
for col in str_cols:
    if col in tousu_stats.columns:
        # tousu_stats[col] = tousu_stats[col].astype('string').fillna('')  # 填充空值为空字符串
        tousu_stats[col] = tousu_stats[col].astype('string')

# 日期字段
date_cols = [ '投诉日期' ]
for col in date_cols:
    if col in tousu_stats.columns:
        tousu_stats[col] = pd.to_datetime(tousu_stats[col]).dt.strftime('%Y-%m-%d')

# 整数字段
int_cols = [ ]
for col in int_cols:
    if col in tousu_stats.columns:
        tousu_stats[col] = pd.to_numeric(tousu_stats[col], errors='coerce').astype('Int64')  # 使用Pandas 的 Int64（可空整数）/ NumPy 的 int64（普通整数）

file_path = r"D:\9.极限给额专题分析\投诉专项分析.xlsx"
write_dataframe_to_excel_com(file_path, {"重渠投诉": tousu_stats}, start_row=1, include_header=True)

成功写入工作表: 重渠投诉
文件已保存: D:\9.极限给额专题分析\投诉专项分析.xlsx
